An example notebook for running the training on all folds. The results were stored and saved on Google Drive.

In [ ]:
mount_drive = 1
if mount_drive:
    from google.colab import drive
    drive.mount('/content/Gdrive')

Mounted at /content/Gdrive


In [ ]:
import sys
!git clone https://github.com/Ignas12345/masters_project_helper_functions.git
sys.path.append('/content/masters_project_helper_functions')

Cloning into 'masters_project_helper_functions'...
remote: Enumerating objects: 196, done.
remote: Counting objects: 100% (27/27), done.
remote: Compressing objects: 100% (18/18), done.
remote: Total 196 (delta 15), reused 20 (delta 9), pack-reused 169 (from 1)
Receiving objects: 100% (196/196), 75.70 KiB | 4.21 MiB/s, done.
Resolving deltas: 100% (114/114), done.


In [ ]:
import pandas as pd
from IPython.display import display
import ast
from sklearn import base
import os

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC, LinearSVC
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, ExtraTreesClassifier
from sklearn.calibration import CalibratedClassifierCV

import masters_project_helper_functions.utils as utils
import masters_project_helper_functions.plotting as plotting
import masters_project_helper_functions.preprocessing_methods as pp
import masters_project_helper_functions.feature_selection as feature_selection
import masters_project_helper_functions.classification_pipeline as classification_pipeline

In [ ]:
url_TCGA_mirna_raw_counts = "https://raw.githubusercontent.com/Ignas12345/masters_project_data_and_notebooks/refs/heads/main/Data/expression_values/TCGA_TGCT_mirna_isoform_data_raw_counts.csv"
url_TCGA_mirna_rpm_counts = "https://raw.githubusercontent.com/Ignas12345/masters_project_data_and_notebooks/refs/heads/main/Data/expression_values/TCGA_TGCT_mirna_isoform_data_rpm.csv"
url_TCGA_TGCT_divisions_by_experiment = "https://raw.githubusercontent.com/Ignas12345/masters_project_data_and_notebooks/refs/heads/main/Data/sample_annotations/TCGA_TGCT_divisions_by_experiment.csv"
fold_path = 'https://raw.githubusercontent.com/Ignas12345/masters_project_data_and_notebooks/refs/heads/main/Data/bootstrap_folds/'

TCGA_TGCT_divisions_by_experiment = pd.read_csv(url_TCGA_TGCT_divisions_by_experiment, index_col=0)
TCGA_mirna_rpm_counts = utils.format_data_frame(url_TCGA_mirna_rpm_counts, sep = ';', decimal= ',', transpose = True).loc[TCGA_TGCT_divisions_by_experiment.index]

df = utils.initial_pre_processing_pipeline(TCGA_mirna_rpm_counts)
experiment_label_dict = TCGA_TGCT_divisions_by_experiment.to_dict()
fold_path = 'https://raw.githubusercontent.com/Ignas12345/masters_project_data_and_notebooks/refs/heads/main/Data/bootstrap_folds/'

reading df from url: https://raw.githubusercontent.com/Ignas12345/masters_project_data_and_notebooks/refs/heads/main/Data/expression_values/TCGA_TGCT_mirna_isoform_data_rpm.csv
nan values filled with 0
column names truncated using: slice(None, 15, None)
df transposed
final shape of df: (156, 3689)

initial shape: (137, 3689)
shape after collapsing: (137, 2212)
initial shape: (137, 2212)
shape after filtering: (137, 2191)


In [ ]:
pre_processing_methods = {'initial_feature_filtering' : pp.feature_filtering_by_class_means,
                          'sample_wise_scaling' : pp.normalize_by_housekeeping_list,
                          'feature_wise_scaling' : pp.log_normalization,
                          'rank_features_and_keep_top_n' : pp.rank_features_and_keep_top_n_features,}
feature_selection_method = feature_selection.rfecv_feature_selection

feature_ranking_method = pp.perform_RFE_ranking

classification_method = LogisticRegression
possible_parameters = [{'C' : 100, 'class_weight' : 'balanced'}]

kwargs = {'housekeeping_list' : ['hsa-mir-191, mature,MIMAT0000440', ],
          'scale_housekeep_by_mean' : False,
          'feature_ranking_method' : feature_ranking_method,
          'keep_n_ranked_features' : 5,
          #'drop_correlated_features': True
          }

kwargs['classification_method_parameters'] = possible_parameters[0]

experiment_name = 'seminoma_vs_embryonal'
sample_label_dict = experiment_label_dict[experiment_name]
df = df.copy()
save_path = f'/content/Gdrive/MyDrive/Magistro_projektas/results_with_cv/Log_reg_C_100_rfe_max_feat_5/{experiment_name}_'
if not os.path.exists(save_path):
  os.makedirs(save_path)

results_df = classification_pipeline.prepare_and_run_pipeline_on_folds(df, fold_path=fold_path, experiment_name=experiment_name, sample_label_dict=sample_label_dict, pre_processing_methods=pre_processing_methods,
                                                                       feature_selection_method = feature_selection_method, classification_method = classification_method, fold_indices = [1877,], save_path = save_path, **kwargs)

In [ ]:
pre_processing_methods = {'initial_feature_filtering' : pp.feature_filtering_by_class_means,
                          'sample_wise_scaling' : pp.normalize_by_housekeeping_list,
                          'feature_wise_scaling' : pp.log_normalization,
                          #'rank_features_and_keep_top_n' : pp.rank_features_and_keep_top_n_features,
                          }
feature_selection_method = feature_selection.select_all_features

#feature_ranking_method = pp.perform_RFE_ranking

classification_method = AdaBoostClassifier
possible_parameters = [{}]

kwargs = {'housekeeping_list' : ['hsa-mir-191, mature,MIMAT0000440', ],
          'scale_housekeep_by_mean' : False,
          #'feature_ranking_method' : feature_ranking_method,
          #'keep_n_ranked_features' : 5,
          #'drop_correlated_features': True
          }

kwargs['classification_method_parameters'] = possible_parameters[0]

for experiment_name in ['seminoma_vs_non_seminoma', 'seminoma_vs_embryonal', 'embryonal_vs_non_embryonal_non_seminoma']:
  sample_label_dict = experiment_label_dict[experiment_name]
  df = df.copy()
  save_path = f'/content/Gdrive/MyDrive/Magistro_projektas/results_with_cv/Ada_boost/{experiment_name}'
  if not os.path.exists(save_path):
    os.makedirs(save_path)

  results_df = classification_pipeline.prepare_and_run_pipeline_on_folds(df, fold_path=fold_path, experiment_name=experiment_name, sample_label_dict=sample_label_dict, pre_processing_methods=pre_processing_methods,
                                                                       feature_selection_method = feature_selection_method, classification_method = classification_method,  save_path = save_path, **kwargs)